# BinX AI & ML Internship
## Week 4 - Day 2: Cross-Validation

Topics covered:
- Why cross-validation beats a single validation split
- How k-fold works: rotating folds
- cross_val_score: mean and standard deviation
- Stratified k-fold for balanced classification folds

---

## 1. Where I'm Starting From

Day 1 established one discipline:

```text
Training   → the model learns here
Validation → I make all decisions here
Test       → opened once at the very end
```

But Day 1 left one question open.

In the learning notebook, the single validation split gave:

```text
Validation F1 = 0.674
```

Is that number reliable? What if I had split the data differently
and ended up with a different 154 rows in the validation set?

```text
Split A → Validation F1 = 0.674
Split B → Validation F1 = 0.712
Split C → Validation F1 = 0.641
```

Same model. Same dataset. Three different answers — depending on
which rows happened to land in validation.

That's the problem cross-validation solves.

---

## 2. Why One Validation Split Can Mislead

A single validation set is just one slice of the data.
If that slice happens to be easier or harder than average,
my estimate of model performance shifts accordingly — not because
the model changed, but because the data did.

This is especially risky with smaller datasets. The Pima dataset
has 768 rows. After a 60/20/20 split, validation holds only 154 rows.
A handful of unusual cases in those 154 rows can swing the F1
score noticeably.

The fix isn't to choose a "better" split — there's no way to know
which split is representative before the fact. The fix is to use
multiple splits and average the results.

---

## 3. What Cross-Validation Does

Cross validation replaces one validation split with several of them.
It divides the training data into k equal parts ("folds"), then trains
and validates k times  each time using a different fold as validation
and the remaining k-1 folds for training.

Averaging the k scores gives a more stable estimate than any single
split could.

One critical point: cross-validation works entirely within the training
portion of the data. The test set stays locked, exactly as it was in Day 1.

```text
Full Dataset
     ↓
Train portion (80%) + Test set (20%)  ← test locked immediately
     ↓
5-Fold CV runs on Train portion only
     ↓
5 fold scores → Mean ± Std
     ↓
Final model → evaluated once on Test
```

This is the key difference from Day 1:

```text
Day 1                          Day 2
──────────────────────         ──────────────────────
Train      → 460 rows          Train/CV   → 614 rows
Validation → 154 rows               ↓
Test       → 154 rows        5 rotating folds
                               validation happens here
                               Test       → 154 rows 
```

In Day 1, I needed a separate validation set to make decisions.
In Day 2, the CV folds handle validation internally — so I pass the
full 80% to cross validation instead of carving out a fixed slice.

---

## 4. K-Fold Cross Validation

With k = 5, the training data is divided into 5 equal folds.
The model trains and validates 5 times, rotating which fold
serves as validation:

| Round | Trains On       | Validates On |
|-------|----------------|--------------|
| 1     | Folds 2,3,4,5  | Fold 1       |
| 2     | Folds 1,3,4,5  | Fold 2       |
| 3     | Folds 1,2,4,5  | Fold 3       |
| 4     | Folds 1,2,3,5  | Fold 4       |
| 5     | Folds 1,2,3,4  | Fold 5       |

The key property: **every sample is used for validation exactly once**,
and for training k-1 times. No data is wasted, and no single split
dominates the result.

---

## 5. Why This Is More Reliable Than a Single Split

Single validation split:

```text
One split → one score → could be lucky or unlucky
```

5-fold cross-validation:

```text
Fold 1 → score 1
Fold 2 → score 2
Fold 3 → score 3
Fold 4 → score 4
Fold 5 → score 5
          ↓
      Average
          ↓
  More stable estimate
```

Instead of saying "my model's F1 is 0.674", I can say:

```text
my model's mean F1 across 5 folds is 0.593 ± 0.036
```

The second statement is a more stable estimate because it is based
on multiple validation folds rather than one split.

---

## 6. Mean and Standard Deviation of CV Scores

`cross_val_score` returns one score per fold. Two numbers
summarize what those scores say together:

**Mean** — the average performance across the validation folds,
giving a more stable estimate of model performance.

**Standard Deviation** — how much the score varies across folds.
Low std means the model performs consistently regardless of which
rows land in validation. High std means performance is sensitive
to the specific split.

```text
Model A — scores: [0.70, 0.71, 0.69, 0.70, 0.71]
  Mean = 0.702   Std = 0.007   → consistent across folds

Model B — scores: [0.55, 0.85, 0.62, 0.91, 0.58]
  Mean = 0.702   Std = 0.160   → same average, but highly variable
```

Both models have the same mean — but Model B's high std means
its performance depends heavily on which fold it gets.
Model A is generally the more reliable choice.

> High mean + low std = generally a strong and stable result.

> High mean + high std = promising average, but performance varies.

---

## 7. Stratified K-Fold

Plain k-fold splits data randomly. With imbalanced classes,
random splitting can create folds with very different class
proportions by chance:

```text
Pima dataset: 65% no diabetes / 35% diabetes

Plain k-fold might produce:
  Fold 1 → 70% / 30%
  Fold 2 → 58% / 42%
  Fold 3 → 68% / 32%
```

Stratified k-fold preserves the original class distribution
in every fold:

```text
Stratified k-fold:
  Fold 1 → 65% / 35%
  Fold 2 → 65% / 35%
  Fold 3 → 65% / 35%
```

This matters because a fold that accidentally has too few diabetic
cases will produce an F1 score that doesn't reflect the real
difficulty of the task.

When `cross_val_score` receives an integer `cv` for a classifier,
Scikit-learn uses StratifiedKFold by default. Here, I'll make the
choice explicit by creating a `StratifiedKFold` object myself —
so I know exactly what's happening rather than relying on a
hidden default.

---

## 8. cross_val_score — Putting It Into Practice

Now that the concept is clear, here's the implementation.

###  Imports Libraries

In [74]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

SEED = 42

### Loading the Data

Same Pima Indians Diabetes dataset — 768 samples,
binary classification (diabetic = 1, not diabetic = 0).

In [75]:
# 1. Define column names
# The original Pima CSV file does not contain column names,
# so we define them manually.

col_names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome"
]

# 2. Load the dataset
# Read the CSV file using pandas.
# header=None:
#   The CSV file does not have a header row.
# names=col_names:
#   Use the column names we defined above.

df = pd.read_csv(
    "../../Week3/Day 3/Data/pima-indians-diabetes.csv",
    header=None,
    names=col_names
)

# 3. Separate Features (X) and Target (y)

# X contains all input features.
# We remove "Outcome" because it is the value we want
# the model to predict.

X = df.drop("Outcome", axis=1)


# y contains the target variable:
# 0 → No Diabetes
# 1 → Diabetes

y = df["Outcome"]

# 4. Check the dataset size

# df.shape[0] → number of rows
# df.shape[1] → number of columns
#
# We subtract 1 because one column (Outcome) is the target,
# so the remaining columns are the features.

print(
    f"Dataset: {df.shape[0]} rows, "
    f"{df.shape[1] - 1} features"
)


# 5. Check the class balance

# Since Outcome contains only 0 and 1,
# the mean of y gives us the proportion of 1s.
#
# For example:
# y = [0, 0, 1, 0, 1]
# mean = 2 / 5 = 0.40 → 40% diabetic
#
# :.1% formats the result as a percentage with 1 decimal place.

print(
    f"Class balance: {y.mean():.1%} diabetic"
)

Dataset: 768 rows, 8 features
Class balance: 34.9% diabetic


### Creating the Train / Test Split

In Day 2, I only need one split — train and test.
The CV folds handle validation internally within the training portion,
so there's no need to carve out a separate fixed validation set.

The test set is locked immediately and stays untouched.

In [76]:
# Creating the Train / Test Split


# We only create ONE fixed split:
#
# 80% → Training portion
#        Cross-Validation will run here
#
# 20% → Test set
#        Locked and untouched until the final evaluation

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    
    # Keep 20% of the data for the final test set
    test_size=0.2,
    
    # Make the split reproducible
    random_state=SEED,
    
    # Preserve the class distribution in both sets
    stratify=y
)


# Check the number of samples


print(
    f"Training portion: {X_train.shape[0]} rows  => CV runs here"
)

print(
    f"Test set:         {X_test.shape[0]} rows   => still locked"
)

print()


# Check class balance

# Because Outcome is binary (0/1),
# the mean represents the proportion of diabetic samples.

print(
    f"Class balance in train: {y_train.mean():.3f}"
)

print(
    f"Class balance in test:  {y_test.mean():.3f}"
)

Training portion: 614 rows  => CV runs here
Test set:         154 rows   => still locked

Class balance in train: 0.349
Class balance in test:  0.351


### Setting Up Stratified K-Fold

I create a `StratifiedKFold` object explicitly rather than passing
`cv=5` as an integer — this way I control exactly how the folds
are built and can see that stratification is applied.

`shuffle=True` randomizes the data before splitting into folds,
so the fold boundaries don't depend on the original row order.

In [77]:
# Create a StratifiedKFold object.

# This will split the TRAINING portion into 5 folds
# while preserving the class distribution in each fold.

skf = StratifiedKFold(

    # Number of folds
    n_splits=5,

    # Shuffle the data before creating the folds
    shuffle=True,

    # Make the shuffling reproducible
    random_state=SEED
)

### Verifying Stratification Across Folds

Before running cross-validation, I want to confirm that each fold
actually preserves the original class balance (34.9% diabetic).

In [78]:
print(f"Original class balance: {y_train.mean():.3f} diabetic\n")

for fold, (_, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    
    # Get the target values for this fold's validation portion
    fold_y = y_train.iloc[val_idx]
    
    # Since Outcome is binary (0/1), the mean = proportion of diabetic cases
    rate = fold_y.mean()
    
    print(f"  Fold {fold}: {rate:.3f} diabetic")

Original class balance: 0.349 diabetic

  Fold 1: 0.350 diabetic
  Fold 2: 0.350 diabetic
  Fold 3: 0.350 diabetic
  Fold 4: 0.350 diabetic
  Fold 5: 0.344 diabetic


### Running cross_val_score

`cross_val_score` handles all the splitting, training,
and scoring internally — one call replaces the manual loop
I would otherwise write for each fold.

Each parameter:
- `model` → the estimator to evaluate
- `X_train, y_train` → the data CV rotates through
- `cv=skf` → the StratifiedKFold splitter defined above
- `scoring="f1"` → the metric computed on each validation fold

In [79]:
# Create the Random Forest model

model = RandomForestClassifier(
    # Limit the depth of each decision tree
    max_depth=3,

    # Build 100 decision trees
    n_estimators=100,

    # Make the model reproducible
    random_state=SEED
)

# Run 5-Fold Stratified Cross-Validation


# cross_val_score automatically:
# 1. Creates the training/validation folds
# 2. Trains the model on each training fold
# 3. Evaluates it on the validation fold
# 4. Calculates the F1 score
# 5. Returns one score for each fold

scores = cross_val_score(
    model,
    X_train,
    y_train,

    # Use the StratifiedKFold object we created earlier
    cv=skf,

    # Calculate F1 on each validation fold
    scoring="f1"
)


# Display the F1 score for each fold

print("F1 score per fold:")

for i, score in enumerate(scores, 1):

    print(
        f"  Fold {i}: {score:.3f}"
    )

F1 score per fold:
  Fold 1: 0.567
  Fold 2: 0.543
  Fold 3: 0.647
  Fold 4: 0.613
  Fold 5: 0.595


The scores vary across folds because each fold contains different
observations. This variation is expected — it's exactly what the
std will quantify in the next step.

### Mean and Standard Deviation

In [80]:
# Calculate the average F1 score across all folds
print(f"Mean F1: {scores.mean():.3f}")

# Calculate how much the F1 score varies across folds
print(f"Std F1:  {scores.std():.3f}")

print()

# Report the result as Mean ± Std
print(f"Reported as: {scores.mean():.3f} ± {scores.std():.3f}")

Mean F1: 0.593
Std F1:  0.036

Reported as: 0.593 ± 0.036


---

## Comparing to the Day 1 Single-Split Score

In Day 1, the single validation split gave F1 = 0.674.
Cross validation gives 0.593 ± 0.036.

The gap of ~0.08 points doesn't mean the model got worse.
It means the Day 1 validation split happened to be an
easier than-average slice of the data   it gave an optimistic
estimate without any way to know that at the time.

Cross validation spread the evaluation across 5 different slices
and averaged them. The 0.593 ± 0.036 is the more stable estimate
to rely on during development.

---

## 9. What Cross-Validation Does NOT Do

Cross-validation is a better way to estimate performance
during development. It is not:

-  A replacement for the final test set
-  Permission to peek at the test set during development
-  Five separate final evaluations

It answers one question:
**"How reliably does this model perform across different
validation folds of the training data?"**

The test set still answers a different question:
**"How does the finalized model perform on data it has
never seen in any form?"**

Both are needed. Neither replaces the other.

---

## What I Learned Today

Day 1 showed that a single train/test split isn't enough —
I need a validation set to make decisions without contaminating
the test set.

Day 2 showed that a single validation set isn't always enough
either — one split can be lucky or unlucky.

Cross-validation replaces that one split with k rotating splits,
averages the results, and adds a standard deviation that shows
how consistent the estimate is.

The workflow going forward:

```text
Full dataset
     ↓
Train / Test split  ← test locked immediately
     ↓
5-Fold Stratified CV on training data → Mean ± Std
     ↓
Final model → evaluated once on Test
```

During development, I'll report mean ± std across folds
instead of relying on a single validation score.